# INITIAL COMMIT

Imports

In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

Reading the data

In [31]:
df = pd.read_csv("data/UNSW_NB15_training-set.csv")
df

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.090200,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.000300,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.005100,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.660800,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.002500,...,1,3,0,0,0,2,3,0,Normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82327,82328,0.000005,udp,-,INT,2,0,104,0,200000.005100,...,1,2,0,0,0,2,1,0,Normal,0
82328,82329,1.106101,tcp,-,FIN,20,8,18062,354,24.410067,...,1,1,0,0,0,3,2,0,Normal,0
82329,82330,0.000000,arp,-,INT,1,0,46,0,0.000000,...,1,1,0,0,0,1,1,1,Normal,0
82330,82331,0.000000,arp,-,INT,1,0,46,0,0.000000,...,1,1,0,0,0,1,1,1,Normal,0


In [32]:
df["proto"].unique()

array(['udp', 'arp', 'tcp', 'igmp', 'ospf', 'sctp', 'gre', 'ggp', 'ip',
       'ipnip', 'st2', 'argus', 'chaos', 'egp', 'emcon', 'nvp', 'pup',
       'xnet', 'mux', 'dcn', 'hmp', 'prm', 'trunk-1', 'trunk-2',
       'xns-idp', 'leaf-1', 'leaf-2', 'irtp', 'rdp', 'netblt', 'mfe-nsp',
       'merit-inp', '3pc', 'idpr', 'ddp', 'idpr-cmtp', 'tp++', 'ipv6',
       'sdrp', 'ipv6-frag', 'ipv6-route', 'idrp', 'mhrp', 'i-nlsp', 'rvd',
       'mobile', 'narp', 'skip', 'tlsp', 'ipv6-no', 'any', 'ipv6-opts',
       'cftp', 'sat-expak', 'ippc', 'kryptolan', 'sat-mon', 'cpnx', 'wsn',
       'pvp', 'br-sat-mon', 'sun-nd', 'wb-mon', 'vmtp', 'ttp', 'vines',
       'nsfnet-igp', 'dgp', 'eigrp', 'tcf', 'sprite-rpc', 'larp', 'mtp',
       'ax.25', 'ipip', 'aes-sp3-d', 'micp', 'encap', 'pri-enc', 'gmtp',
       'ifmp', 'pnni', 'qnx', 'scps', 'cbt', 'bbn-rcc', 'igp', 'bna',
       'swipe', 'visa', 'ipcv', 'cphb', 'iso-tp4', 'wb-expak', 'sep',
       'secure-vmtp', 'xtp', 'il', 'rsvp', 'unas', 'fc', 'iso-ip',


### Feature engineering:
#### Dropping certain columns of the data
- "id" is an index and does not help with categorization
- "attack_cat" is an extension of a label and should not be included in the training data

In [33]:
df.drop(["id", "attack_cat"], axis=1, inplace=True)

#### One Hot Encoding columns with non-numerical data:
1. "proto" - the column denoting the protocol of the packet
2. "state" - state of the packet
3. "service" - 

In [34]:
# create map of groups to protocol
protocol_groups = {
    'common_transport': ['tcp', 'udp', 'sctp'],
    
    'routing': ['ospf', 'eigrp', 'egp', 'igp', 'nsfnet-igp', 'dgp', 'idrp', 'idpr', 'idpr-cmtp', 'sdrp', 'mhrp'],
    
    'tunneling': ['gre', 'ipip', 'l2tp', 'encap', 'etherip', 'mobile', 'ipcomp', 'ipnip', 'ip', 'micp'],
    
    'ipv6_family': ['ipv6', 'ipv6-frag', 'ipv6-route', 'ipv6-no', 'ipv6-opts'],
    
    'multicast': ['igmp', 'pim', 'vrrp', 'pgm', 'cbt'],
    
    'link_layer': ['arp', 'ax.25', 'fc', 'srp', 'il', 'ipx-n-ip'],
    
    'security': ['skip', 'tlsp', 'rsvp', 'kryptolan', 'secure-vmtp', 'aes-sp3-d', 'swipe', 'pri-enc'],
    
    'legacy': ['ggp', 'st2', 'argus', 'chaos', 'nvp', 'pup', 'xnet', 'mux', 'dcn', 'hmp', 'prm', 'trunk-1', 'trunk-2', 'xns-idp', 'leaf-1', 'leaf-2', 'irtp', 'rdp', 'netblt', 'mfe-nsp', 'merit-inp', '3pc', 'ddp', 'tp++', 'narp', 'any', 'cftp', 'sat-expak', 'ippc', 'sat-mon', 'cpnx', 'wsn', 'pvp', 'br-sat-mon', 'sun-nd', 'wb-mon', 'vmtp', 'ttp', 'vines', 'tcf', 'sprite-rpc', 'larp', 'mtp', 'bbn-rcc', 'bna', 'visa', 'ipcv', 'cphb', 'iso-tp4', 'wb-expak', 'sep', 'xtp', 'unas', 'iso-ip', 'aris', 'a/n', 'snp', 'compaq-peer', 'zero', 'ddx', 'iatp', 'stp', 'uti', 'sm', 'smp', 'isis', 'ptp', 'fire', 'crtp', 'crudp', 'sccopmce', 'iplt', 'pipe', 'sps', 'ib', 'emcon', 'gmtp', 'ifmp', 'pnni', 'qnx', 'scps']
}

# create a mapping of protocols to group
proto_to_group = {proto: group for group, protos in protocol_groups.items() for proto in protos}

# map protocol to their respective group
df['proto_group'] = df['proto'].map(proto_to_group).fillna('legacy')


In [36]:
# one hot encode protocol group
df_ohe = pd.get_dummies(df, columns=['proto_group', 'state', 'service'])

# convert boolean columns to int
bool_cols = df_ohe.select_dtypes(include='bool').columns
df_ohe[bool_cols] = df_ohe[bool_cols].astype(int)
df_ohe

,dur,proto,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,...,service_ftp,service_ftp-data,service_http,service_irc,service_pop3,service_radius,service_smtp,service_snmp,service_ssh,service_ssl
0,0.000011,udp,2,0,496,0,90909.090200,254,0,1.803636e+08,...,0,0,0,0,0,0,0,0,0,0
1,0.000008,udp,2,0,1762,0,125000.000300,254,0,8.810000e+08,...,0,0,0,0,0,0,0,0,0,0
2,0.000005,udp,2,0,1068,0,200000.005100,254,0,8.544000e+08,...,0,0,0,0,0,0,0,0,0,0
3,0.000006,udp,2,0,900,0,166666.660800,254,0,6.000000e+08,...,0,0,0,0,0,0,0,0,0,0
4,0.000010,udp,2,0,2126,0,100000.002500,254,0,8.504000e+08,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82327,0.000005,udp,2,0,104,0,200000.005100,254,0,8.320000e+07,...,0,0,0,0,0,0,0,0,0,0
82328,1.106101,tcp,20,8,18062,354,24.410067,254,252,1.241044e+05,...,0,0,0,0,0,0,0,0,0,0
82329,0.000000,arp,1,0,46,0,0.000000,0,0,0.000000e+00,...,0,0,0,0,0,0,0,0,0,0
82330,0.000000,arp,1,0,46,0,0.000000,0,0,0.000000e+00,...,0,0,0,0,0,0,0,0,0,0


Drop the "proto" column

In [ ]:
df_ohe.drop(columns=["proto"], inplace=True)

#### Normalizing numerical columns

In [40]:
df_ohe.columns

Index(['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
       'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
       'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
       'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'label',
       'proto_group_common_transport', 'proto_group_ipv6_family',
       'proto_group_legacy', 'proto_group_link_layer', 'proto_group_multicast',
       'proto_group_routing', 'proto_group_security', 'proto_group_tunneling',
       'state_ACC', 'state_CLO', 'state_CON', 'state_FIN', 'state_INT',
       'state_REQ', 'state_RST', 'service_-', 'service_dhcp', 'service_dns',
       'service_ftp', 'service_ftp-data', 'service_http', 'service_irc',
       'service_pop3', 'service_radiu